# Minimal RLM Spark log root-cause analysis

Create a small Spark log, ask `fabric_rlm` to find the root cause, and display the payload.

In [ ]:
%pip install -q fabric-rlm==0.4.1

In [ ]:
from pathlib import Path

log_path = "/tmp/spark_app.log"
Path(log_path).write_text(
    """
24/01/01 10:00:01 INFO DAGScheduler: Got job 17 collect at SparkApp.scala:142
24/01/01 10:01:30 WARN TaskSetManager: Lost task 88001.0 in stage 42.0 on worker-07.fabric.local executor 7: java.lang.OutOfMemoryError
24/01/01 10:02:31 WARN TaskSetManager: Lost task 88001.1 in stage 42.0 on worker-07.fabric.local executor 7: java.lang.OutOfMemoryError
24/01/01 10:03:32 WARN TaskSetManager: Lost task 88001.2 in stage 42.0 on worker-07.fabric.local executor 7: java.lang.OutOfMemoryError
24/01/01 10:04:33 ERROR TaskSetManager: Task 88001 in stage 42.0 failed 4 times; aborting job
24/01/01 10:04:34 ERROR DAGScheduler: Job 17 failed: collect at SparkApp.scala:142, took 1247.58 s
24/01/01 10:04:35 ERROR YarnScheduler: Lost executor 7 on worker-07.fabric.local: java.lang.OutOfMemoryError: Java heap space
24/01/01 10:04:36 INFO SparkContext: Successfully stopped SparkContext
    """.strip(),
    encoding="utf-8",
)
log_path

In [ ]:
from fabric_rlm import FabricLM, File, RLM

rlm = RLM.task(
    task="Analyze the Spark log. Return failed_job_id, failed_stage_id, failing_executor, root_cause, and evidence.",
    inputs={"log_file": File(log_path)},
    outputs=["failed_job_id", "failed_stage_id", "failing_executor", "root_cause", "evidence"],
    lm=FabricLM("gpt-5.1", reasoning_effort="low"),
    skills=["data_exploration"],
    max_turns=6,
)

result = rlm.run()
result.payload